# Building a primitive agent using python
* Before using a library, we code an agent from scratch
* Tutorial link : https://www.youtube.com/watch?v=vHDwpoSFdQY&t=314s
  
## Step 1: initialize API keys

In [1]:
!pip install -qU huggingface_hub

## Step 2: Initialize the client
* Point the client to your local Ollama server instead of the cloud router
* add the local Ollama endpoint and a dummy key

In [1]:
from huggingface_hub import InferenceClient

# Point the client to your local Ollama server instead of the cloud router
client = InferenceClient(
    base_url="http://localhost:11434/v1",  # Local Ollama endpoint
    api_key="ollama"                    # Dummy key (Ollama doesn't require a real one)
)

In [119]:
response=client.chat.completions.create(
    model="qwen2.5:3b",
    messages=[
        {"role":"user", "content":"are you  sentient?"} ],
    extra_body={"keep_alive": 0} # Unloads model instantly from RAM
     
)

In [121]:
import pprint
pprint.pprint(response.choices[0].message.content)


('No, I am not sentient. I am an artificial intelligence created to process '
 'and generate text based on patterns and data I have been trained on, but I '
 'do not possess consciousness, emotions, or intentions.')


## Note
* QWEN 2.5 :3b is not a reasoning model so it lacks the think tags
* Next, try to use deepseek in your personal study on this topic:  model="deepseek-r1:1.5b"

In [125]:
pprint.pprint(response.choices[0].message)#chech reasoning/ thinking tokens


ChatCompletionMessage(content='No, I am not sentient. I am an artificial intelligence created to process and generate text based on patterns and data I have been trained on, but I do not possess consciousness, emotions, or intentions.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


## Step 3: Lets create the tool for the Agent to use  it to get tasks done

In [197]:
## create a simple weather function
def get_weather(city):
    """
    Get the weather in a given city.
    """
    c = city.lower()
    if c == "chicago":
        return f"The current weather in {city} is cloudy."
    if c == "arizona":
        return f"The current weather in {city} is sunny."
    if c == "singapore":
        return f"The current weather in {city} is rainy."
    return f"The current weather in {city} is sunny."

## Step 4: Give tools to the agent
* Createe a schema telling the agent the  tools it has available
* Learn about a library called pydantic: a validation model. Its an easier way to create a schema(@14 mins into the video).
* But now we do it manually

In [189]:
get_weather_tool_schema={
    "type":"function",
    "function":{"name":"get_weather",
                "description":"Get the current temperature in a given city.",
                "parameters":{"type":"object",
                              "properties": {
                                  "city":{
                                      "type":"string",
                                      "description":"The city to get the weather for.Add only the name of the city"
                                              }},
                              "required":["city"]
                             }
               }
}

In [146]:
#Giving my model the get_weather tool
response=client.chat.completions.create(
    model="qwen2.5:3b",
    messages= [{"role":"user", "content":"What is the weather in Singapore today?"} ],
    tools=[get_weather_tool_schema],
    tool_choice="auto" #lets the model decide when to call functions
)

In [152]:
response.choices[0].message.content

''

In [154]:
#For a reasoning model  use: response.choices[0].message.reasoning_content
#to see the agent call a function tool
import pprint
pprint.pprint(response.choices[0].message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_s230ijkj', function=Function(arguments='{"city":"Singapore"}', name='get_weather'), type='function', index=0)]


In [156]:

response.choices[0].message.tool_calls[0].function.__dict__


{'arguments': '{"city":"Singapore"}', 'name': 'get_weather'}

* **The LLM received our querry and generated a call to the get_weather tool**
* **The agent received the tool call and executed the get_weather tool but we need it to go back to the LLM to generate a response**

## Create an Agent that takes the LLM JSON block and generates a response
Create a wrapper agent class that:
* Enables communication between agent and user by using a loop
* Helps the agent recognise when it needs to call a tool from  an LLM json block
* Enables itself to run the tools
* Enables it to show results when it gets a final answer 

In [199]:
import json

class Agent:
    # 1. initialize/set up an agent class 
    def __init__(self, client: InferenceClient, system: str = "", tools: list = None) -> None:
        self.client = client # takes my LLM client
        self.system = system # system prompt (optional)
        self.messages: list = [] # empty message history list, diff messages stored here
        self.tools = tools if tools is not None else []
        
        # Ensures system prompt is added ONCE at the top of message history
        if self.system:
            self.messages.append({"role": "system", "content": self.system})

    # 2. defining how we call an agent, with a message eg: agent("what is the weather in Kiambu?")
    def __call__(self, message=""):
        # Append the new user message to the conversation history
        if message:
            self.messages.append({"role": "user", "content": message})

        # execute agent with the message: 
        # call LLM, verify for tool calls, execute tools, return results to llm and save here
        final_assistant_content = self.execute()

        # if final answer is returned, store it to list of messages in memory
        if final_assistant_content:
            self.messages.append({"role": "assistant", "content": final_assistant_content}) 
        
        return final_assistant_content

    # 3. execution loop
    def execute(self):
        # initialize execute so that the agent only breaks out of a loop when it has an answer
        while True:
            completion = self.client.chat.completions.create(
                model="qwen2.5:3b",
                messages=self.messages, # every message in message list/history
                tools=self.tools,
                tool_choice="auto", # lets the model decide when to call functions
                extra_body={"keep_alive": 0} # clears RAM immediately after call
            )
            response_message = completion.choices[0].message
            
            # verify existence of tool calls
            if response_message.tool_calls:
                self.messages.append(response_message)

                tool_outputs = []
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)

                    # If there need be a tool execution, run this:
                    if function_name in globals() and callable(globals()[function_name]):
                        function_to_call = globals()[function_name]
                        executed_output = function_to_call(**function_args)
                        tool_output_content = str(executed_output) # Ensure output is string
                        print(f"Executing tool: {function_name} with args {function_args}, Output: {tool_output_content[:500]}...")

                    tool_outputs.append(
                        {
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "name": function_name,
                            "content": tool_output_content,
                        }
                    )

                self.messages.extend(tool_outputs) # tool outputs will be available in message history

            else: # if there is no tool to call, return the final answer
                return response_message.content

## Agent demo

In [201]:
#initialize the client
client = InferenceClient(
    base_url="http://localhost:11434/v1",  # Local Ollama endpoint
    api_key="ollama"                    # Dummy key (Ollama doesn't require a real one)
)

system="You are a helpful assistant that can answer questions using the provided tools."
tools=[get_weather_tool_schema]
agent=Agent(client, system, tools)

In [203]:
agent("What is the weather in Singapore?")

Executing tool: get_weather with args {'city': 'Singapore'}, Output: The current weather in Singapore is rainy....


'The current weather in Singapore is rainy.'

In [205]:
agent.messages

[{'role': 'system',
  'content': 'You are a helpful assistant that can answer questions using the provided tools.'},
 {'role': 'user', 'content': 'What is the weather in Singapore?'},
 ChatCompletionOutputMessage(role='assistant', content='', reasoning=None, tool_call_id=None, tool_calls=[ChatCompletionOutputToolCall(function=ChatCompletionOutputFunctionDefinition(arguments='{"city":"Singapore"}', name='get_weather', description=None), id='call_kugrzbsx', type='function', index=0)]),
 {'tool_call_id': 'call_kugrzbsx',
  'role': 'tool',
  'name': 'get_weather',
  'content': 'The current weather in Singapore is rainy.'},
 {'role': 'assistant',
  'content': 'The current weather in Singapore is rainy.'}]

In [207]:
agent.system

'You are a helpful assistant that can answer questions using the provided tools.'

In [209]:
agent.tools
     

[{'type': 'function',
  'function': {'name': 'get_weather',
   'description': 'Get the current temperature in a given city.',
   'parameters': {'type': 'object',
    'properties': {'city': {'type': 'string',
      'description': 'The city to get the weather for.Add only the name of the city'}},
    'required': ['city']}}}]

In [215]:
!git init


Reinitialized existing Git repository in C:/Users/User/AI Agents Huggingface/.git/


In [219]:
# Add all files in the current directory
!git add .

# Commit with a message
!git commit -m "Add local AI agent notebook and detailed README"

# Set branch name to main and push (force push '-u origin main' if it's a fresh repo setup)
!git branch -M main
!git push -u origin main

[main c7b3961] Add local AI agent notebook and detailed README
 7 files changed, 4274 insertions(+)
 create mode 100644 ToolCallingAgents.ipynb
 rename Autoformatting_tool.ipynb => Unit 1/Autoformatting_tool.ipynb (100%)
 create mode 100644 Unit 1/Dummy_Agent1.ipynb
 create mode 100644 Unit 1/README.md
 rename Tokenizing_1.0.ipynb => Unit 1/Tokenizing_1.0.ipynb (100%)
 create mode 100644 Untitled.ipynb
 create mode 100644 codeAgents.ipynb
branch 'main' set up to track 'origin/main'.


To https://github.com/GreenArcher4/Hugging-face-AI-agent.git
   63a91fc..c7b3961  main -> main
